# Road Centerline Segmentation - Massachusetts Roads Dataset

Trains a small U-Net to predict road **centerlines** (not filled road-surface area) from
1m/px aerial RGB tiles, then exports it to ONNX so it can plug straight into
`backend/road_extraction.py`'s `arcgispro-py3` environment (which already uses
`onnxruntime` for the oil-palm YOLO model - no need to add PyTorch there).

**Why this dataset**: [Massachusetts Roads Dataset](https://www.kaggle.com/datasets/balraj98/massachusetts-roads-dataset)
is 1m/px - the same resolution class as this project's own real orthophoto
(`260726_Bypass AKT_1m.tif`, see README's Road/Trail Extraction accuracy section) - and its
masks are already rasterized road *centerlines* (7px-wide OSM lines), matching what we
actually want the model to output, unlike DeepGlobe's filled-road-surface masks.

**On Kaggle**: New Notebook -> Add Input -> search "Massachusetts Roads Dataset"
(balraj98) -> Settings -> Accelerator -> GPU T4 x2 (or P100). Then Run All.

**After this notebook**: the exported `.onnx` is a pretrained *base* model, not a
finished one - see the last section for fine-tuning it on your own digitized data
(`hasil digit/digitasi jalan.shp` + its source orthophoto) before using it for real.


## 1. Setup

In [ ]:
import glob
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CROP_SIZE = 512          # training patch size, cropped from each 1500x1500 tile
BATCH_SIZE = 8
EPOCHS = 25              # raise if your Kaggle session has time left - checkpoints
                          # every epoch below so you can stop early without losing progress
LR = 1e-3
OUT_DIR = "/kaggle/working"


## 2. Find the dataset

Kaggle's exact folder naming has drifted before between dataset versions, so this
searches for the images/labels folders instead of hardcoding a path - if it can't find
them, it prints what *is* under `/kaggle/input/` so you can fix `DATA_ROOT` by hand in
one place instead of hunting through every cell below.


In [ ]:
def _find_dir(root, name_contains):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if name_contains in d.lower():
                yield os.path.join(dirpath, d)

INPUT_ROOT = "/kaggle/input"
candidates = [d for d in glob.glob(f"{INPUT_ROOT}/*massachusetts*roads*")] or \
             [d for d in glob.glob(f"{INPUT_ROOT}/*")]
if not candidates:
    raise FileNotFoundError(
        "No dataset attached - Add Input -> 'Massachusetts Roads Dataset' (balraj98) first."
    )
DATA_ROOT = candidates[0]
print("DATA_ROOT:", DATA_ROOT)

train_img_dir = next(_find_dir(DATA_ROOT, "train"), None)
train_lbl_dir = next((d for d in _find_dir(DATA_ROOT, "train") if "label" in d.lower()), None)
val_img_dir = next((d for d in _find_dir(DATA_ROOT, "val") if "label" not in d.lower()), None)
val_lbl_dir = next((d for d in _find_dir(DATA_ROOT, "val") if "label" in d.lower()), None)

# Re-scan: the first "train"-containing dir found above might itself be train_labels -
# separate them explicitly by whether "label" is in the name.
all_dirs = list(_find_dir(DATA_ROOT, ""))
train_img_dir = next((d for d in all_dirs if d.lower().endswith("train")), train_img_dir)
train_lbl_dir = next((d for d in all_dirs if "train" in d.lower() and "label" in d.lower()), train_lbl_dir)
val_img_dir = next((d for d in all_dirs if d.lower().endswith("val")), val_img_dir)
val_lbl_dir = next((d for d in all_dirs if "val" in d.lower() and "label" in d.lower()), val_lbl_dir)

print("train images:", train_img_dir)
print("train labels:", train_lbl_dir)
print("val images:  ", val_img_dir)
print("val labels:  ", val_lbl_dir)

if not all([train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]):
    print("\nCouldn't auto-detect all four folders - here's what's actually under "
          f"{DATA_ROOT}, set the four *_dir variables above by hand:")
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        depth = dirpath[len(DATA_ROOT):].count(os.sep)
        if depth > 3:
            continue
        print("  " * depth + os.path.basename(dirpath) + "/",
              f"({len(filenames)} files)" if filenames else "")


## 3. Dataset

In [ ]:
def _list_images(d):
    exts = (".tif", ".tiff", ".png", ".jpg", ".jpeg")
    return sorted(f for f in os.listdir(d) if f.lower().endswith(exts))

def _match_label(img_name, label_dir):
    # Image/label pairs share a filename stem in this dataset (e.g. 10078660_15.tiff /
    # 10078660_15.tif) - match by stem instead of assuming identical extensions.
    stem = os.path.splitext(img_name)[0]
    for f in os.listdir(label_dir):
        if os.path.splitext(f)[0] == stem:
            return f
    return None


class RoadDataset(Dataset):
    """Random CROP_SIZE crop per item (re-cropped fresh each epoch via __getitem__,
    not precomputed) - cheap augmentation for free, and avoids duplicating the dataset
    on disk as a separate tiled copy."""

    def __init__(self, img_dir, label_dir, crop_size, train):
        self.img_dir, self.label_dir = img_dir, label_dir
        self.crop_size = crop_size
        self.train = train
        self.names = _list_images(img_dir)
        self.pairs = [(n, _match_label(n, label_dir)) for n in self.names]
        self.pairs = [(i, l) for i, l in self.pairs if l is not None]
        if not self.pairs:
            raise RuntimeError(f"No matching image/label pairs found in {img_dir} / {label_dir}")
        print(f"{img_dir}: {len(self.pairs)} image/label pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, lbl_name = self.pairs[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        lbl = Image.open(os.path.join(self.label_dir, lbl_name)).convert("L")

        w, h = img.size
        cs = self.crop_size
        x0 = random.randint(0, max(0, w - cs))
        y0 = random.randint(0, max(0, h - cs))
        img = img.crop((x0, y0, x0 + cs, y0 + cs))
        lbl = lbl.crop((x0, y0, x0 + cs, y0 + cs))

        img_arr = np.array(img, dtype=np.float32) / 255.0
        lbl_arr = (np.array(lbl, dtype=np.float32) > 127).astype(np.float32)

        if self.train:
            if random.random() < 0.5:
                img_arr, lbl_arr = img_arr[:, ::-1].copy(), lbl_arr[:, ::-1].copy()
            if random.random() < 0.5:
                img_arr, lbl_arr = img_arr[::-1, :].copy(), lbl_arr[::-1, :].copy()
            k = random.randint(0, 3)
            img_arr, lbl_arr = np.rot90(img_arr, k).copy(), np.rot90(lbl_arr, k).copy()

        img_t = torch.from_numpy(img_arr).permute(2, 0, 1)   # (3, H, W)
        lbl_t = torch.from_numpy(lbl_arr).unsqueeze(0)        # (1, H, W)
        return img_t, lbl_t


train_ds = RoadDataset(train_img_dir, train_lbl_dir, CROP_SIZE, train=True)
val_ds = RoadDataset(val_img_dir, val_lbl_dir, CROP_SIZE, train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


## 4. Model - small U-Net

No `segmentation_models_pytorch` dependency (Kaggle images don't reliably ship it) - a compact from-scratch U-Net, plenty for a single-class (road/not-road) task.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """Standard 4-level U-Net, base width 32 (kept small on purpose - this dataset is
    modest in size and Kaggle GPU sessions are time-boxed; widen CHANNELS if you have
    time/data to spare)."""

    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        chs = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc1, self.enc2, self.enc3, self.enc4 = (
            DoubleConv(in_ch, chs[0]), DoubleConv(chs[0], chs[1]),
            DoubleConv(chs[1], chs[2]), DoubleConv(chs[2], chs[3]),
        )
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(chs[3], chs[4])
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2)
        self.dec4 = DoubleConv(chs[4], chs[3])
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2)
        self.dec3 = DoubleConv(chs[3], chs[2])
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2)
        self.dec2 = DoubleConv(chs[2], chs[1])
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2)
        self.dec1 = DoubleConv(chs[1], chs[0])
        self.out = nn.Conv2d(chs[0], out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)   # raw logits - apply sigmoid outside (BCEWithLogits below)


model = UNet().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")


## 5. Loss and metric

BCE + Dice combined - plain BCE alone struggles here because road pixels are a small
minority of each tile (heavy class imbalance); Dice directly rewards getting the thin
road class right regardless of how rare it is.


In [ ]:
def dice_loss(logits, target, eps=1e-6):
    probs = torch.sigmoid(logits)
    probs, target = probs.flatten(1), target.flatten(1)
    intersection = (probs * target).sum(1)
    union = probs.sum(1) + target.sum(1)
    return 1 - ((2 * intersection + eps) / (union + eps)).mean()

bce = nn.BCEWithLogitsLoss()

def criterion(logits, target):
    return bce(logits, target) + dice_loss(logits, target)

@torch.no_grad()
def iou_score(logits, target, thr=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > thr).float()
    intersection = (preds * target).sum()
    union = ((preds + target) > 0).float().sum()
    return ((intersection + eps) / (union + eps)).item()


## 6. Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
# torch.amp (not the older torch.cuda.amp) - newer PyTorch (Kaggle's default image)
# deprecated the cuda-specific spelling in favor of this device-agnostic one.
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "val_iou": []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss, val_iou = 0.0, 0.0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, lbls).item() * imgs.size(0)
            val_iou += iou_score(logits, lbls) * imgs.size(0)
    val_loss /= len(val_ds)
    val_iou /= len(val_ds)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_iou={val_iou:.4f}")

    # Checkpoint every epoch (not just at the end) - a Kaggle session that gets cut off
    # partway through still leaves the best-so-far weights in /kaggle/working.
    torch.save(model.state_dict(), f"{OUT_DIR}/unet_last.pth")
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), f"{OUT_DIR}/unet_best.pth")

print("best val IoU:", best_val_iou)


## 7. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(history["val_iou"])
axes[1].set_title("Val IoU"); axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=120)
plt.show()


## 8. Visualize a few predictions

In [ ]:
model.load_state_dict(torch.load(f"{OUT_DIR}/unet_best.pth"))
model.eval()

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row in range(3):
    idx = random.randrange(len(val_ds))
    img_t, lbl_t = val_ds[idx]
    with torch.no_grad():
        pred = torch.sigmoid(model(img_t.unsqueeze(0).to(device)))[0, 0].cpu().numpy()
    axes[row, 0].imshow(img_t.permute(1, 2, 0).numpy()); axes[row, 0].set_title("image")
    axes[row, 1].imshow(lbl_t[0].numpy(), cmap="gray"); axes[row, 1].set_title("ground truth")
    axes[row, 2].imshow(pred > 0.5, cmap="gray"); axes[row, 2].set_title("prediction")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/sample_predictions.png", dpi=120)
plt.show()


## 9. Export to ONNX

`opset=13` matches what `road_extraction.py`'s sibling model (`sawit_detector.onnx`,
YOLOv8) already targets in this project - keeps both models compatible with the same
`onnxruntime` version already installed in the `arcgispro-py3` conda env, no separate
opset-compatibility check needed later.


In [ ]:
model.load_state_dict(torch.load(f"{OUT_DIR}/unet_best.pth"))
model.eval().cpu()

dummy = torch.zeros(1, 3, CROP_SIZE, CROP_SIZE)
onnx_path = f"{OUT_DIR}/road_unet.onnx"
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {2: "height", 3: "width"}, "logits": {2: "height", 3: "width"}},
    opset_version=13,
)
print("exported:", onnx_path)

# Sanity check it actually loads and runs before you download it.
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
test_out = sess.run(None, {"image": dummy.numpy()})[0]
print("ONNX output shape:", test_out.shape)


## 10. Next steps (not run here)

1. **Download** `/kaggle/working/road_unet.onnx` (Kaggle notebook output panel) into
   `backend/` alongside `sawit_detector.onnx`.
2. **Fine-tune on your own data** before trusting it on real sites: this model has only
   ever seen Massachusetts roads. Upload your own orthophoto tile(s) +
   `hasil digit/digitasi jalan.shp` (rasterized to a centerline mask the same way this
   notebook's labels are shaped) as a private Kaggle dataset, load `unet_best.pth` here,
   drop the learning rate (e.g. `LR = 1e-4`), and continue training a handful of epochs
   on just that data - standard transfer-learning fine-tuning, much more realistic than
   training from scratch on a single ~6km road.
3. **Wire it into `road_extraction.py`**: add an `onnxruntime.InferenceSession` call
   (same pattern as `yolo_detector.py`'s `_session_get()`) that predicts a road-pixel
   probability mask per raster block, in place of (or blended with) `land_clearing.py`'s
   ExG threshold - then the existing `skimage.morphology.skeletonize` +
   `arcpy.conversion.RasterToPolyline` + `_drop_short_bridges` pipeline downstream stays
   unchanged, since it only cares about getting *a* binary road mask, not how it was
   produced. Re-run the same accuracy check (README's Road/Trail Extraction section,
   `hasil digit/digitasi jalan.shp` vs. its source orthophoto) to see whether this
   actually beats the current ExG-threshold F1 (60.3%) before replacing it for real.
